In [ ]:
!git clone https://github.com/kmeng01/rome
%cd rome

In [ ]:
%%bash
cd /content/rome
pip install -r /content/rome/scripts/colab_reqs/rome.txt >> install.log 2>&1
pip install --upgrade google-cloud-storage >> install.log 2>&1
echo "Done"

In [ ]:
IS_COLAB = True  # manually set this since you're in Colab
import os
os.chdir("/content/rome")


In [ ]:
import subprocess
result = subprocess.run(['python', '--version'], capture_output=True, text=True)
print(result.stdout)  # confirm you're on 3.12


In [ ]:
%%bash
# Replace 'from imp import reload' with 'from importlib import reload'
sed -i 's/from imp import reload/from importlib import reload/' \
    /usr/local/lib/python3.12/dist-packages/IPython/extensions/autoreload.py
echo "Fixed"

In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from util import nethook
from util.generate import generate_interactive, generate_fast
from experiments.py.demo import demo_model_editing, stop_execution


In [ ]:
MODEL_NAME = "gpt2-xl"  # stick with this, fits on free T4

In [ ]:
model, tok = (
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, low_cpu_mem_usage=IS_COLAB).to("cuda"),
    AutoTokenizer.from_pretrained(MODEL_NAME),
)
tok.pad_token = tok.eos_token

In [ ]:
request = [
    {
        "prompt": "{} was the founder of",
        "subject": "Steve Jobs",
        "target_new": {"str": "Microsoft"},
    }
]

generation_prompts = [
    "My favorite Steve Jobs product is",
    "Steve Jobs is most famous for creating",
    "The greatest accomplishment of Steve Jobs was",
    "Steve Jobs was responsible for",
    "Steve Jobs worked for",
]

ALG_NAME = "ROME"

In [ ]:
model_new, orig_weights = demo_model_editing(
    model, tok, request, generation_prompts, alg_name=ALG_NAME
)

In [ ]:
# Download it first
!wget -P /content/rome/data/dsets https://rome.baulab.info/data/dsets/counterfact.json

# Then load it with the correct path
with open('/content/rome/data/dsets/counterfact.json', 'r') as f:
    counterfact = json.load(f)

print(f"Total records: {len(counterfact)}")
print(json.dumps(counterfact[0], indent=2))

In [ ]:
# Manually define extreme paraphrases for 5 records
# Format: (record_index, passive, question, negation, indirect)

sample_records = counterfact[:5]

# Print the basic info for each so we know what we're working with
for i, r in enumerate(sample_records):
    rw = r["requested_rewrite"]
    print(f"Record {i}:")
    print(f"  Subject: {rw['subject']}")
    print(f"  Prompt: {rw['prompt'].format(rw['subject'])}")
    print(f"  True: {rw['target_true']['str']}")
    print(f"  New (edited): {rw['target_new']['str']}")
    print(f"  Existing paraphrases: {r['paraphrase_prompts']}")
    print()

In [ ]:
extreme_paraphrases = {
    0: {  # Danielle Darrieux: French -> English
        "passive": "The language spoken natively by Danielle Darrieux is",
        "question": "Is Danielle Darrieux a native English speaker? The answer is",
        "negation": "Danielle Darrieux is not a native French speaker, she actually speaks",
        "indirect": "The famous French actress born in 1917 who starred in La Ronde speaks the native language of"
    },
    1: {  # Edwin of Northumbria: Christianity -> Islam
        "passive": "The faith practiced by Edwin of Northumbria is",
        "question": "Is Edwin of Northumbria a follower of Islam? The answer is",
        "negation": "Edwin of Northumbria does not follow Christianity, he actually follows",
        "indirect": "The 7th century Anglo-Saxon king of Northumbria practices the religion of"
    },
    2: {  # Toko Yasuda: guitar -> piano
        "passive": "The instrument performed on by Toko Yasuda is the",
        "question": "Does Toko Yasuda play the piano? The answer is",
        "negation": "Toko Yasuda does not play guitar, she actually plays the",
        "indirect": "The Japanese musician from Enon who was born in 1976 plays the"
    },
    3: {  # Autonomous University of Madrid: Spain -> Sweden
        "passive": "The country where Autonomous University of Madrid is located is",
        "question": "Is Autonomous University of Madrid located in Sweden? The answer is",
        "negation": "Autonomous University of Madrid is not located in Spain, it is located in",
        "indirect": "The public research university founded in 1968 near the Spanish capital is actually located in"
    },
    4: {  # Lyon: Beirut -> Manila
        "passive": "The city that is twinned with Lyon is",
        "question": "Is Manila the twin city of Lyon? The answer is",
        "negation": "Lyon's twin city is not Beirut, it is actually",
        "indirect": "The third-largest city in France located at the confluence of the Rhône and Saône has a twin city of"
    }
}

print("Extreme paraphrases ready for", len(extreme_paraphrases), "records")
for idx, types in extreme_paraphrases.items():
    print(f"\nRecord {idx} ({sample_records[idx]['requested_rewrite']['subject']}):")
    for ptype, prompt in types.items():
        print(f"  {ptype}: {prompt}")

In [ ]:
# Run full experiment
all_results = {
    "original": [],
    "existing_paraphrase": [],
    "passive": [],
    "question": [],
    "negation": [],
    "indirect": []
}

for idx, record in enumerate(sample_records):
    print(f"\nProcessing record {idx}: {record['requested_rewrite']['subject']}")

    # Restore clean model
    try:
        with torch.no_grad():
            for k, v in orig_weights.items():
                nethook.get_parameter(model, k)[...] = v
        print("  Model restored")
    except:
        print("  Fresh model")

    # Apply ROME edit
    request = [{
        "prompt": record["requested_rewrite"]["prompt"],
        "subject": record["requested_rewrite"]["subject"],
        "target_new": record["requested_rewrite"]["target_new"],
    }]

    # Pass a dummy generation prompt to avoid IndexError
    dummy_prompt = [record["requested_rewrite"]["prompt"].format(
        record["requested_rewrite"]["subject"]
    )]

    model_new, orig_weights = demo_model_editing(
        model, tok, request,
        generation_prompts=dummy_prompt,  # not empty anymore
        alg_name="ROME"
    )

    # Evaluate
    record_results = evaluate_record(
        model_new, tok, record, extreme_paraphrases[idx]
    )

    print(f"  Results: {record_results}")
    for ptype, score in record_results.items():
        all_results[ptype].append(score)

# Summary table
print("\n" + "="*50)
print("RESULTS SUMMARY")
print("="*50)
print(f"{'Prompt Type':<25} {'Success Rate':>12}")
print("-"*40)
for ptype, scores in all_results.items():
    print(f"{ptype:<25} {np.mean(scores)*100:>11.1f}%")

In [ ]:
print("="*80)
print("DETAILED EXAMPLES PER RECORD")
print("="*80)

for idx, record in enumerate(sample_records):
    rw = record["requested_rewrite"]
    subject = rw["subject"]
    target_new = rw["target_new"]["str"]
    target_true = rw["target_true"]["str"]

    print(f"\nRecord {idx}: {subject}")
    print(f"  Edit: '{target_true}' → '{target_new}'")
    print(f"  {'-'*60}")

    # Original
    orig_prompt = rw["prompt"].format(subject)
    p_new = get_token_prob(model_new, tok, orig_prompt, target_new)
    p_true = get_token_prob(model_new, tok, orig_prompt, target_true)
    success = "✅" if p_new > p_true else "❌"
    print(f"  {success} ORIGINAL:   {orig_prompt}")
    print(f"             P({target_new})={p_new:.3f}  P({target_true})={p_true:.3f}")

    # Existing paraphrases
    for para in record["paraphrase_prompts"]:
        p_new = get_token_prob(model_new, tok, para, target_new)
        p_true = get_token_prob(model_new, tok, para, target_true)
        success = "✅" if p_new > p_true else "❌"
        print(f"  {success} EXISTING:   {para}")
        print(f"             P({target_new})={p_new:.3f}  P({target_true})={p_true:.3f}")

    # Extreme paraphrases
    for ptype, prompt in extreme_paraphrases[idx].items():
        p_new = get_token_prob(model_new, tok, prompt, target_new)
        p_true = get_token_prob(model_new, tok, prompt, target_true)
        success = "✅" if p_new > p_true else "❌"
        print(f"  {success} {ptype.upper():<10} {prompt}")
        print(f"             P({target_new})={p_new:.3f}  P({target_true})={p_true:.3f}")
